# Week 9: multiresolution analysis

**What we do.** We study the filters behind a multiresolution analysis (MRA): the Haar low-pass filter, the high-pass filter obtained from it as a quadrature mirror filter, the conditions an orthonormal filter must satisfy, and frequency responses of Daubechies filters. We also see what the Haar MRA does to a signal: projections onto the spaces `V_j` of piecewise constant functions.

**What to expect.** The low-pass response `|m0(ω)|²` equals 1 at `ω = 0` and 0 at `ω = π`, and `|m0(ω)|² + |m0(ω + π)|² = 1` for every orthonormal filter. Longer Daubechies filters separate low and high frequencies more sharply.

**How to read the code.** Filters are short NumPy arrays `h[n]`, normalized so that `Σ h[n] = sqrt(2)` (the `rec_lo` filters of PyWavelets). Functions come from `wavelets_lab.mra`; we also use the `pywt.Wavelet` objects of PyWavelets directly.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pywt

from wavelets_lab import mra

## Haar filters

**What we do.** We get the Haar low-pass filter `h`, build the high-pass filter `g` from it and check the orthonormality conditions.

**What to expect.** `h = [1, 1]/sqrt(2)` (averaging), `g = [1, -1]/sqrt(2)` (differencing), and `True` for the check.

**How to read the code.** `mra.quadrature_mirror(h)` reverses `h` and alternates the signs; `mra.is_orthonormal_filter(h)` checks `Σ h = sqrt(2)` and `Σ h[n] h[n + 2k] = δ_k`.

In [ ]:
h = mra.haar_filter()
g = mra.quadrature_mirror(h)
h, g, mra.is_orthonormal_filter(h)

## Haar approximations of a signal

**What we do.** We approximate a smooth curve by its Haar projections `P_j f` onto `V_j`: functions constant on intervals of length `2^-j`. For the Haar MRA the projection is simply the average of `f` over each interval.

**What to expect.** Staircases that get finer and closer to the curve as `j` grows. The difference between two consecutive levels, `P_{j+1} f - P_j f`, is what the wavelet coefficients of level `j` describe (the details).

**How to read the code.** The signal is sampled at 1024 points. For level `j` we reshape it into `2^j` blocks (`f.reshape(2**j, -1)`), average each block (`.mean(axis=1)`) and repeat each mean over its block (`np.repeat`). This is exact Haar MRA on the sampled signal, done with plain NumPy.

In [ ]:
x = np.linspace(0.0, 1.0, 1024, endpoint=False)
f = np.sin(2 * np.pi * x) + 0.5 * np.sin(6 * np.pi * x) * (x > 0.5)
plt.figure(figsize=(10, 4))
plt.plot(x, f, "k", label="f")
for level in (2, 4, 6):
    blocks = f.reshape(2**level, -1)
    projection = np.repeat(blocks.mean(axis=1), blocks.shape[1])
    plt.plot(x, projection, label=f"P_{level} f (intervals of length 2^-{level})")
plt.xlabel("x")
plt.ylabel("value")
plt.legend()
plt.show()

## Frequency responses

**What we do.** We plot `|m0(ω)|²` of the Haar filter and the Daubechies filters db2, db4 and db8 on `[0, π]`, and the sum `|m0(ω)|² + |m0(ω + π)|²` for db4.

**What to expect.** All curves start at 1 and end at 0, i.e. they are low-pass filters; the longer the filter, the closer the curve is to an ideal step at `π/2` (better frequency separation). The sum in the second panel is exactly 1 (the quadrature mirror condition, equivalent to orthonormality).

**How to read the code.** `mra.wavelet_filters(name)` returns the PyWavelets filters `(rec_lo, rec_hi)`; `mra.frequency_response(h, omega)` evaluates `m0(ω) = (1/sqrt 2) Σ h[n] e^{-inω}`.

In [ ]:
omega = np.linspace(0.0, np.pi, 400)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name in ("haar", "db2", "db4", "db8"):
    low, _ = mra.wavelet_filters(name)
    axes[0].plot(omega, np.abs(mra.frequency_response(low, omega)) ** 2, label=name)
axes[0].set_xlabel("frequency ω [rad]")
axes[0].set_ylabel("|m0(ω)|²")
axes[0].legend()
low, _ = mra.wavelet_filters("db4")
total = (
    np.abs(mra.frequency_response(low, omega)) ** 2
    + np.abs(mra.frequency_response(low, omega + np.pi)) ** 2
)
axes[1].plot(omega, total)
axes[1].set_ylim(0.9, 1.1)
axes[1].set_xlabel("frequency ω [rad]")
axes[1].set_ylabel("|m0(ω)|² + |m0(ω + π)|²")
axes[1].set_title("db4: quadrature mirror condition")
plt.tight_layout()
plt.show()

## Low-pass and high-pass together

**What we do.** We plot the responses of the db4 low-pass filter `h` and its quadrature mirror `g`.

**What to expect.** The two curves are mirror images of each other around `π/2` and cross at the value 1/2: `h` keeps the lower half of the frequencies, `g` the upper half. This is the "two-band" split repeated at every level of the wavelet transform.

**How to read the code.** We evaluate both filters with `mra.frequency_response` (which uses the same normalization `1/sqrt 2` for `g`).

In [ ]:
low, high = mra.wavelet_filters("db4")
plt.figure(figsize=(7, 4))
plt.plot(omega, np.abs(mra.frequency_response(low, omega)) ** 2, label="low-pass h")
plt.plot(omega, np.abs(mra.frequency_response(high, omega)) ** 2, label="high-pass g")
plt.xlabel("frequency ω [rad]")
plt.ylabel("squared response")
plt.legend()
plt.show()

## Wavelet objects in PyWavelets

**What we do.** We look at the PyWavelets description of the Daubechies wavelet db3 and check that its high-pass filter is the quadrature mirror of its low-pass filter.

**What to expect.** `rec_lo` and `rec_hi` have 6 coefficients each, db3 has 3 vanishing moments, and the check returns `True`.

**How to read the code.** `pywt.Wavelet("db3")` is an object with attributes `rec_lo`, `rec_hi` (reconstruction filters), `dec_lo`, `dec_hi` (decomposition filters, reversed) and `vanishing_moments_psi`. `np.allclose` compares arrays up to rounding.

In [ ]:
wavelet = pywt.Wavelet("db3")
wavelet.rec_lo, wavelet.rec_hi, wavelet.vanishing_moments_psi

In [ ]:
np.allclose(mra.quadrature_mirror(wavelet.rec_lo), wavelet.rec_hi)

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** One module is used: `wavelets_lab.mra` (`src/wavelets_lab/mra.py`).

**How to read the code.** A filter is a 1D array `h` with `Σ h = sqrt(2)`; `ω` is a frequency in radians per sample (`π` is the highest frequency a sampled signal can have).

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `pywt.Wavelet(name)` | description of a wavelet: filters, vanishing moments, … | name → object |
| `array.reshape(rows, -1)` | reshapes; `-1` means "as many as needed" | array → array |
| `array.mean(axis=1)` | mean of every row | 2D array → 1D array |
| `np.repeat(a, k)` | repeats every entry `k` times | array → longer array |
| `np.allclose(a, b)` | equality up to rounding | arrays → `bool` |

### Module `wavelets_lab.mra` (file `src/wavelets_lab/mra.py`)

Multiresolution analysis (MRA): filters, the construction of Daubechies wavelets and the cascade algorithm. A **low-pass filter** `h` is the sequence of the scaling equation `φ(x) = sqrt(2) sum_n h[n] φ(2x - n)`, normalized so that `sum_n h[n] = sqrt(2)`; this is the `rec_lo` filter of PyWavelets. Its frequency response is `m0(ω) = (1/sqrt 2) sum_n h[n] e^{-inω}`. Imports:
```python
from math import comb
import numpy as np
import pywt
```
and the constant `SQRT2 = sqrt(2)`.

#### `mra.haar_filter()`

**What it is.** The low-pass filter of the Haar MRA, `h = [1, 1] / sqrt(2)`: its scaling function is the indicator of `[0, 1)`, which satisfies `φ(x) = φ(2x) + φ(2x - 1)`.

**Input.** None.

**Output.** `float64` array `[0.7071…, 0.7071…]`.

**How to write it yourself.** Return `np.array([1.0, 1.0]) / np.sqrt(2)`.

#### `mra.quadrature_mirror(h)`

**What it is.** The high-pass (wavelet) filter of an orthogonal MRA, `g[n] = (-1)^n h[L - 1 - n]`: the low-pass filter reversed, with alternating signs. With it, `ψ(x) = sqrt(2) sum_n g[n] φ(2x - n)` is the wavelet. For PyWavelets filters it equals `rec_hi`.

**Input.** `h` (array-like): low-pass filter of even length `L`.

**Output.** `float64` array of length `L`.

**How to write it yourself.**
1. `low = np.asarray(h, dtype=np.float64)`.
2. Signs `(-1.0) ** np.arange(L)`.
3. Return `signs * low[::-1]`.

#### `mra.is_orthonormal_filter(h, atol=1e-10)`

**What it is.** Checks the two conditions on the filter of an orthonormal MRA: `sum_n h[n] = sqrt(2)` (so `m0(0) = 1`, the filter passes constants) and `sum_n h[n] h[n + 2k] = δ_k` (orthonormality of the integer shifts of `φ`, equivalent to `|m0(ω)|^2 + |m0(ω + π)|^2 = 1`).

**Input.** `h` (array-like), `atol` (`float`).

**Output.** `bool`.

**How to write it yourself.**
1. Full autocorrelation `r = np.correlate(h, h, mode="full")`; entry `L - 1 + j` is `sum_n h[n] h[n + j]`.
2. Keep even lags `j = 0, 2, 4, …`: `r[L - 1 :: 2]`.
3. Expected values: 1 for lag 0, 0 for the others.
4. Return `np.isclose(h.sum(), sqrt(2), atol=atol) and np.allclose(even_lags, expected, atol=atol)`.

#### `mra.wavelet_filters(name)`

**What it is.** Returns the reconstruction filters of a PyWavelets wavelet, to compare own constructions with the library.

**Input.** `name` (`str`): e.g. `"haar"`, `"db3"`, `"sym4"`.

**Output.** Tuple `(h, g)` of `float64` arrays: `rec_lo` (low-pass) and `rec_hi` (high-pass).

**How to write it yourself.**
1. `wavelet = pywt.Wavelet(name)`.
2. Return `np.asarray(wavelet.rec_lo)`, `np.asarray(wavelet.rec_hi)`.

#### `mra.frequency_response(h, omega)`

**What it is.** Evaluates the filter's frequency response `m0(ω) = (1/sqrt 2) sum_n h[n] e^{-inω}`. For a low-pass MRA filter `m0(0) = 1` and `m0(π) = 0`.

**Input.** `h` (array-like): filter; `omega` (array-like): frequencies in radians.

**Output.** Complex array of the values `m0(ω)`.

**How to write it yourself.**
1. Matrix `E = np.exp(-1j * np.multiply.outer(omega, np.arange(len(h))))` (one row per frequency).
2. Return `E @ h / np.sqrt(2)`.